In [ ]:
import logging
from pathlib import Path

import SimpleITK as sitk
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from radiomics import featureextractor
from natsort import natsorted
from tqdm.notebook import tqdm

In [ ]:
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

logger = logging.getLogger("radiomics.featureextractor")
logger.setLevel(logging.ERROR)

In [ ]:
# root = Path('/media/yannik/Intenso/DATA/dfg_plexus/')
root = Path('/run/media/yannik/Intenso/DATA/dfg_plexus/')

# Tabular data
hc_data = pd.read_csv(root / "Dataset002_DFGFinetuned_3d_fullres_full_new_hc___volumes.csv")
patho_data = pd.read_csv(root / "Dataset002_DFGFinetuned_3d_fullres_full_ms___volumes.csv")

# Image and mask data
image_dir_hc = root / "HC_T1s___freesurfer___reorientated_padded/"
print(image_dir_hc.exists())
image_dir_ms = root / "T1s/"
print(image_dir_ms.exists())
pred_dir_hc = root / "nnUNet/nnUNet_inference_results/Dataset002_DFGFinetuned_3d_fullres_full_new_hc/"
print(pred_dir_hc.exists())
pred_dir_ms = root / "nnUNet/nnUNet_inference_results/Dataset002_DFGFinetuned_3d_fullres_full_ms/"
print(pred_dir_ms.exists())

In [ ]:
def load_img_mask(img_path: Path, mask_path: Path, verbose: bool = True):
    # Test SimpleITK loading
    try:
        image = sitk.ReadImage(str(img_path))
        mask = sitk.ReadImage(str(mask_path))
        if verbose:
            img_array = sitk.GetArrayFromImage(image)
            print("Image and mask loaded successfully.")
            print(f"Image: Size: {image.GetSize()} Spacing: {image.GetSpacing()}")
            print(f"Mask: Size: {mask.GetSize()} Spacing: {mask.GetSpacing()}")
            print(f"Image min: {img_array.min()} max: {img_array.max()} mean: {img_array.mean()}")
            # Intensity histogram visualisation
            plt.hist(img_array.flatten(), bins=100, color='blue', alpha=0.5)
            plt.title("Intensity Histogram")
            plt.xlabel("Intensity")
            plt.ylabel("Frequency")
            plt.show()
        return image, mask
    except Exception as e:
        raise ValueError(f"Error loading image or mask with SimpleITK: {e}")

In [ ]:
# Load image and mask
img_path = image_dir_hc / "hc0001_T1.nii"
mask_path = pred_dir_hc / "hc0001.nii.gz"
image, mask = load_img_mask(img_path, mask_path, verbose=True)

In [ ]:
# Visualize image with mask as overlay
def plot_img_mask(image, mask):
    img_array = sitk.GetArrayFromImage(image)
    mask_array = sitk.GetArrayFromImage(mask)

    n_slices = 10
    fig, ax = plt.subplots(2, n_slices, figsize=(2 * n_slices, 4))
    for i, slice in enumerate(np.linspace(60, 100, n_slices)):
        ax[0, i].imshow(img_array[int(slice)], cmap='gray')
        ax[0, i].axis('off')
        ax[1, i].imshow(img_array[int(slice)], cmap='gray')
        ax[1, i].imshow(mask_array[int(slice)], alpha=0.5, cmap='jet')
        ax[1, i].axis('off')
    plt.show()

In [ ]:
plot_img_mask(image, mask)

In [ ]:
img_path = image_dir_ms / "Subject001_T1.nii"
mask_path = pred_dir_ms / "Subject001.nii.gz"
image, mask = load_img_mask(img_path, mask_path, verbose=True)

In [ ]:
plot_img_mask(image, mask)

In [ ]:
mask_data = sitk.GetArrayFromImage(mask)
np.unique(mask_data)

### Feature Extraction - HC

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor("pyradiomics_conf.yaml")

all_features = pd.DataFrame()

for img_path, mask_path in tqdm(zip(natsorted(image_dir_hc.glob("*.nii")), natsorted(pred_dir_hc.glob("*.nii.gz"))),
                                desc='Extracting HC features',
                                total=len(list(image_dir_hc.glob("*.nii")))):

    subject_id = mask_path.stem.removesuffix(".nii")
    features = extractor.execute(str(img_path), str(mask_path))

    features_row = pd.DataFrame([features], index=[subject_id])
    all_features = pd.concat([all_features, features_row], axis=0)

df = all_features.copy()

# Drop diagnostics / metadata columns
diag_cols = [c for c in df.columns if c.startswith("diagnostics_")]
df = df.drop(columns=diag_cols, errors="ignore")

# Keep only radiomics feature columns (for Original image type)
# They start with 'original_' if we use `imageType: Original`
feature_cols = [c for c in df.columns if c.startswith("original_")]
radiomics_df = df[feature_cols].copy()

# Force everything to numeric (anything non-numeric becomes NaN)
radiomics_df = radiomics_df.apply(
    pd.to_numeric,
    errors="coerce"   # invalid values -> NaN instead of breaking
)

radiomics_df.to_csv(root / "radiomics_features_hc.csv", sep=";")

In [ ]:
radiomics_df.shape

### Feature Extraction - MS

In [ ]:
all_features = pd.DataFrame()

for img_path, mask_path in tqdm(zip(natsorted(image_dir_ms.glob("*.nii")), natsorted(pred_dir_ms.glob("*.nii.gz"))),
                                desc='Extracting MS features',
                                total=len(list(image_dir_ms.glob("*.nii")))):

    # print(img_path)

    subject_id = mask_path.stem.removesuffix(".nii")
    features = extractor.execute(str(img_path), str(mask_path))

    features_row = pd.DataFrame([features], index=[subject_id])
    all_features = pd.concat([all_features, features_row], axis=0)

df = all_features.copy()

# Drop diagnostics / metadata columns
diag_cols = [c for c in df.columns if c.startswith("diagnostics_")]
df = df.drop(columns=diag_cols, errors="ignore")

# Keep only radiomics feature columns (for Original image type)
# They start with 'original_' if we use `imageType: Original`
feature_cols = [c for c in df.columns if c.startswith("original_")]
radiomics_df = df[feature_cols].copy()

# Force everything to numeric (anything non-numeric becomes NaN)
radiomics_df = radiomics_df.apply(
    pd.to_numeric,
    errors="coerce"   # invalid values -> NaN instead of breaking
)

radiomics_df.to_csv(root / "radiomics_features_ms.csv", sep=";")